<a href="https://colab.research.google.com/github/MuhammadJawadFasih/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadJawadFasih/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


I will prioritize content pages that have a strong search opportunity but appear to be underperforming.

My baseline rule uses two signals:

1. **Search volume:** Pages with higher search volume receive a higher opportunity score because improving them could affect more searches.
2. **CTR relative to position:** Pages with a reasonable average position but relatively low CTR receive a higher score because their search visibility may not be converting into clicks efficiently.

The rule will combine these signals into one opportunity score and rank pages from highest to lowest.

### Reason code

* **HIGH_VOLUME_LOW_CTR:** The page has meaningful search volume and its CTR is relatively low compared with pages at a similar average position.

The action label for this baseline is:

* **REVIEW_REFRESH:** Review the page for possible content or search-result improvements.

This is a decision-support baseline, not a claim that every selected page definitely needs a refresh.


In [13]:
# ML-07 — Section 1: Signal checks

import pandas as pd
import numpy as np
from pathlib import Path

# Load the starter dataset
DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

# Check required signals
required_columns = ["search_volume", "ctr", "avg_position"]

for col in required_columns:
    print(f"{col}: {col in df.columns}")

# Keep only valid observations for the signal audit
audit = df[required_columns].copy()

audit = audit.replace([np.inf, -np.inf], np.nan)
audit = audit.dropna(subset=required_columns)

# FlyRank data rule:
# avg_position = 0 means no position data
audit = audit[audit["avg_position"] > 0].copy()

print("\nTotal observations used (n):", len(audit))


# =========================================================
# SIGNAL 1: SEARCH VOLUME
# =========================================================

audit["volume_bucket"] = pd.qcut(
    audit["search_volume"],
    q=4,
    duplicates="drop"
)

volume_table = (
    audit.groupby("volume_bucket", observed=True)
    .agg(
        n=("search_volume", "size"),
        median_search_volume=("search_volume", "median"),
        median_ctr=("ctr", "median"),
        median_position=("avg_position", "median")
    )
    .reset_index()
)

print("\nSIGNAL 1 — SEARCH VOLUME")
print(volume_table.to_string(index=False))

print(
    "\nVerdict: CONFIRMED"
)

print(
    "Reason: the buckets show that search volume is a real, "
    "observable opportunity signal. Higher-volume pages represent "
    "more potential search demand."
)


# =========================================================
# SIGNAL 2: CTR VS POSITION
# =========================================================

audit["position_bucket"] = pd.cut(
    audit["avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True
)

ctr_position_table = (
    audit.groupby("position_bucket", observed=True)
    .agg(
        n=("ctr", "size"),
        median_ctr=("ctr", "median"),
        median_position=("avg_position", "median"),
        median_search_volume=("search_volume", "median")
    )
    .reset_index()
)

print("\nSIGNAL 2 — CTR VS AVERAGE POSITION")
print(ctr_position_table.to_string(index=False))

print(
    "\nVerdict: MIXED"
)

print(
    "Reason: CTR varies across position buckets, so CTR should not "
    "be interpreted by itself. A low CTR can be more meaningful when "
    "considered relative to the page's observed search position."
)

Dataset shape: (30000, 44)
search_volume: True
ctr: True
avg_position: True

Total observations used (n): 27076

SIGNAL 1 — SEARCH VOLUME
  volume_bucket     n  median_search_volume  median_ctr  median_position
 (-0.001, 10.0] 18091                   0.0        0.10             11.6
   (10.0, 20.0]  2245                  20.0        0.10             11.1
(20.0, 74000.0]  6740                  90.0        0.05             13.1

Verdict: CONFIRMED
Reason: the buckets show that search volume is a real, observable opportunity signal. Higher-volume pages represent more potential search demand.

SIGNAL 2 — CTR VS AVERAGE POSITION
position_bucket     n  median_ctr  median_position  median_search_volume
            1-3   934        0.03              2.3                  10.0
           4-10 10791        0.16              6.6                  10.0
          11-20  7028        0.11             14.0                  10.0
            21+  8323        0.00             31.1                  10.0

Ve

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*



I will calculate one baseline opportunity score for every content page using the signals defined in Section 1.

The score will prioritize pages with:

* higher search volume, because they represent a larger potential search opportunity;
* lower CTR relative to their average search position, because this can indicate an opportunity to improve how the page attracts clicks.

Pages will be ranked from the highest score to the lowest score.

Each row will receive:

* an **action label**: `REVIEW_REFRESH`
* a **reason code**: `HIGH_VOLUME_LOW_CTR`
* a numeric **baseline score**
* a **rank**

The ranked queue will be written to:

`work/outputs/baseline_action_score.csv`

This baseline uses only the observed signals available in the analysis window and does not use future-window information or product/ground-truth flags.


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-07 — Section 2: Build the ranked queue

import numpy as np
from pathlib import Path

queue = df.copy()

# Make sure the signals are numeric
queue["search_volume"] = pd.to_numeric(
    queue["search_volume"], errors="coerce"
)

queue["ctr"] = pd.to_numeric(
    queue["ctr"], errors="coerce"
)

queue["avg_position"] = pd.to_numeric(
    queue["avg_position"], errors="coerce"
)

# ---------------------------------------------------------
# POSITION BUCKETS
# ---------------------------------------------------------

queue["position_bucket"] = pd.cut(
    queue["avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True
)

# Median CTR for each position bucket
position_ctr_median = (
    queue.loc[queue["avg_position"] > 0]
    .groupby("position_bucket", observed=True)["ctr"]
    .median()
)

queue["position_ctr_median"] = queue["position_bucket"].map(
    position_ctr_median
)

# ---------------------------------------------------------
# TRANSPARENT RULE
# ---------------------------------------------------------

# High-volume threshold:
# use the observed median search volume.
volume_threshold = queue["search_volume"].median()

queue["high_volume"] = (
    queue["search_volume"] >= volume_threshold
)

# Low CTR relative to pages in the same position bucket
queue["low_ctr_vs_position"] = (
    (queue["avg_position"] > 0)
    & queue["position_ctr_median"].notna()
    & (queue["ctr"] <= queue["position_ctr_median"])
)

# Score:
# search volume when BOTH conditions are satisfied.
queue["baseline_score"] = np.where(
    queue["high_volume"] & queue["low_ctr_vs_position"],
    queue["search_volume"],
    0
)

# ---------------------------------------------------------
# REASON CODE + ACTION
# ---------------------------------------------------------

queue["reason_code"] = np.where(
    queue["baseline_score"] > 0,
    "HIGH_VOLUME_LOW_CTR",
    "NO_OPPORTUNITY"
)

queue["action"] = np.where(
    queue["baseline_score"] > 0,
    "REVIEW_REFRESH",
    "NO_ACTION"
)

# ---------------------------------------------------------
# RANK EVERYTHING
# ---------------------------------------------------------

queue = queue.sort_values(
    ["baseline_score", "search_volume"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

# ---------------------------------------------------------
# SELECT OUTPUT COLUMNS
# ---------------------------------------------------------

if "content_id" in queue.columns:
    id_column = "content_id"
elif "content_hash_id" in queue.columns:
    id_column = "content_hash_id"
else:
    id_column = None

output_columns = [
    "rank",
    "baseline_score",
    "action",
    "reason_code",
    "search_volume",
    "ctr",
    "avg_position"
]

if id_column is not None:
    output_columns.insert(1, id_column)

output = queue[output_columns].copy()

# ---------------------------------------------------------
# WRITE REQUIRED CSV
# ---------------------------------------------------------

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

output.to_csv(output_path, index=False)

print("CSV written:", output_path)
print("Rows ranked:", len(output))
print("High-volume threshold:", volume_threshold)

print("\nTop 20:")
print(output.head(20).to_string(index=False))

CSV written: work/outputs/baseline_action_score.csv
Rows ranked: 30000
High-volume threshold: 10.0

Top 20:
 rank           content_id  baseline_score         action         reason_code  search_volume  ctr  avg_position
    1 content_bf67a444faef         60500.0 REVIEW_REFRESH HIGH_VOLUME_LOW_CTR        60500.0 0.00          45.5
    2 content_5ec29ae79c60         60500.0 REVIEW_REFRESH HIGH_VOLUME_LOW_CTR        60500.0 0.00          49.8
    3 content_deb54e9e19cd         60500.0 REVIEW_REFRESH HIGH_VOLUME_LOW_CTR        60500.0 0.00          41.7
    4 content_454cc6654c6e         60500.0 REVIEW_REFRESH HIGH_VOLUME_LOW_CTR        60500.0 0.00          44.9
    5 content_cd6760921db8         49500.0 REVIEW_REFRESH HIGH_VOLUME_LOW_CTR        49500.0 0.00          47.3
    6 content_f76ccf7a7834         49500.0 REVIEW_REFRESH HIGH_VOLUME_LOW_CTR        49500.0 0.15           9.5
    7 content_83e3da1394ac         49500.0 REVIEW_REFRESH HIGH_VOLUME_LOW_CTR        49500.0 0.00          6

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*



I reviewed the top 20 pages selected by the baseline rule.

For each page, I record:

* **Action:** the action assigned by the baseline rule.
* **Reason code:** why the page was selected.
* **Confidence note:** how strong the observed signals are.
* **What would make it wrong:** a condition that could make the recommendation misleading or unsuitable for refresh.

The review is intended as decision-support rather than proof that every selected page requires a refresh.


In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-07 — Section 3: Top-20 review

# Use the actual top 20 generated by Section 2
top20 = output.head(20).copy()

# 90th percentile of search volume for a simple confidence reference
volume_p90 = output["search_volume"].quantile(0.90)

def make_confidence_note(row):
    if row["avg_position"] <= 10 and row["search_volume"] >= volume_p90:
        return "Stronger baseline signal: high search volume and position within 10."

    if row["avg_position"] <= 20:
        return "Moderate confidence: high search volume with a measurable position."

    return "Lower confidence: high search volume, but weak average position."

def make_wrong_note(row):
    if row["avg_position"] > 20:
        return (
            "It could be wrong if the poor position means the page has limited "
            "realistic opportunity for a refresh to improve visibility."
        )

    if row["ctr"] == 0:
        return (
            "It could be wrong if zero CTR reflects low click opportunity, "
            "query intent, or SERP conditions rather than a content problem."
        )

    return (
        "It could be wrong if the low CTR is explained by search intent, "
        "SERP features, or another factor not captured by this baseline."
    )

top20["confidence_note"] = top20.apply(
    make_confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    make_wrong_note,
    axis=1
)

review_columns = [
    "rank",
    "action",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]

if "content_id" in top20.columns:
    review_columns.insert(1, "content_id")

print("TOP-20 REVIEW")
print("=" * 100)

print(
    top20[review_columns].to_string(index=False)
)

TOP-20 REVIEW
 rank           content_id         action         reason_code                                                      confidence_note                                                                                                        what_would_make_it_wrong
    1 content_bf67a444faef REVIEW_REFRESH HIGH_VOLUME_LOW_CTR     Lower confidence: high search volume, but weak average position.    It could be wrong if the poor position means the page has limited realistic opportunity for a refresh to improve visibility.
    2 content_5ec29ae79c60 REVIEW_REFRESH HIGH_VOLUME_LOW_CTR     Lower confidence: high search volume, but weak average position.    It could be wrong if the poor position means the page has limited realistic opportunity for a refresh to improve visibility.
    3 content_deb54e9e19cd REVIEW_REFRESH HIGH_VOLUME_LOW_CTR     Lower confidence: high search volume, but weak average position.    It could be wrong if the poor position means the page has limited realistic

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*



I reviewed the baseline picks for cases where the rule may have selected a page for a weak or misleading reason.

A weak pick would be a page that receives a high score from the rule but does not show a convincing combination of the selected signals. Possible reasons include unusually high search volume without meaningful evidence of a CTR opportunity, or a low CTR that may be explained by the page's search position.

I will treat these cases as limitations of the baseline rather than automatically removing them.

### Leakage check

The baseline uses only signals available in the defined analysis window. I did not use future-window performance, product flags, or any outcome/label information to calculate the score.

The baseline is therefore intended as a simple pre-model benchmark that can later be compared with the Week-5 model.


In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-07 — Section 4: Weak picks + leakage check

print("WEAK PICKS")
print("=" * 80)

# Identify skeptical cases in the top 20:
# very high volume but poor position, or position 1-3 with zero CTR.
weak_picks = top20[
    (top20["avg_position"] > 20) |
    ((top20["avg_position"] <= 3) & (top20["ctr"] == 0))
].copy()

print(f"Weak/skeptical picks found: {len(weak_picks)}")

if len(weak_picks) > 0:
    print("\nExamples:")
    print(
        weak_picks[
            [
                "rank",
                "content_id",
                "search_volume",
                "ctr",
                "avg_position",
                "action",
                "reason_code"
            ]
        ].to_string(index=False)
    )
else:
    print("No skeptical picks were identified by these checks.")


# ---------------------------------------------------------
# LEAKAGE CHECK
# ---------------------------------------------------------

print("\n\nLEAKAGE CHECK")
print("=" * 80)

# These fields are explicitly prohibited from being used
# as baseline features according to the FlyRank data guidance.
forbidden_fields = [
    "trend_pct",
    "trend_direction",
    "is_declining_label",
    "health_score",
    "needs_ctr_fix",
    "is_quick_win"
]

used_features = [
    "search_volume",
    "ctr",
    "avg_position"
]

print("Features used by the baseline:")
for feature in used_features:
    print(f"  - {feature}")

print("\nChecking for forbidden fields in the score inputs:")

used_forbidden = [
    field for field in forbidden_fields
    if field in used_features
]

if len(used_forbidden) == 0:
    print("PASS — no product flags or label-derived fields were used.")
else:
    print("FAIL — forbidden fields detected:", used_forbidden)

assert len(used_forbidden) == 0


# ---------------------------------------------------------
# CHECK THE OUTPUT
# ---------------------------------------------------------

print("\nOutput checks:")

assert len(output) == len(df)
assert output["rank"].is_unique
assert output["rank"].min() == 1
assert output["rank"].max() == len(df)

required_output_columns = [
    "rank",
    "baseline_score",
    "action",
    "reason_code"
]

for col in required_output_columns:
    assert col in output.columns

print("PASS — all 30,000 rows are ranked.")
print("PASS — ranks are unique and complete.")
print("PASS — required score/action/reason columns exist.")
print("PASS — baseline uses only observed search/engagement signals.")

print("\nLeakage conclusion:")
print(
    "The baseline does not use trend labels, product decision flags, "
    "or future outcome fields as score inputs."
)

WEAK PICKS
Weak/skeptical picks found: 10

Examples:
 rank           content_id  search_volume  ctr  avg_position         action         reason_code
    1 content_bf67a444faef        60500.0  0.0          45.5 REVIEW_REFRESH HIGH_VOLUME_LOW_CTR
    2 content_5ec29ae79c60        60500.0  0.0          49.8 REVIEW_REFRESH HIGH_VOLUME_LOW_CTR
    3 content_deb54e9e19cd        60500.0  0.0          41.7 REVIEW_REFRESH HIGH_VOLUME_LOW_CTR
    4 content_454cc6654c6e        60500.0  0.0          44.9 REVIEW_REFRESH HIGH_VOLUME_LOW_CTR
    5 content_cd6760921db8        49500.0  0.0          47.3 REVIEW_REFRESH HIGH_VOLUME_LOW_CTR
    7 content_83e3da1394ac        49500.0  0.0          65.5 REVIEW_REFRESH HIGH_VOLUME_LOW_CTR
    8 content_ee4630879d03        49500.0  0.0          25.2 REVIEW_REFRESH HIGH_VOLUME_LOW_CTR
   10 content_84fe9d0a707a        40500.0  0.0          43.3 REVIEW_REFRESH HIGH_VOLUME_LOW_CTR
   14 content_6b41450ae50c        27100.0  0.0          43.2 REVIEW_REFRESH HIGH_VO

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.